# Kako vpliva povprečna bruto plača na rodnost po regijah v Sloveniji?

In [1]:
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

COLORS = {
    "muted_olive":   "#8CB369",
    "sandy_brown":   "#F4A259",
    "blushed_brick": "#BC4B51",
}

df = pd.read_csv('data/raw/csv/regije_avg_placa.csv', encoding='cp1250', sep=';', skiprows=2, decimal='.')
df.columns = ["regija", "bruto_2023", "bruto_2024", "bruto_2025", "neto_2023", "neto_2024", "neto_2025"]

rod = pd.read_csv('data/raw/csv/regije_rodnost.csv', encoding='cp1250', sep=';', skiprows=2)
rod.columns = ["leto", "regija", "skupaj", "na1000"]
rod["leto"] = rod["leto"].astype(int)

exclude = ["SLOVENIJA", "Vzhodna Slovenija", "Zahodna Slovenija"]
SLO_BRUTO = {2023: 2254.86, 2024: 2394.92, 2025: 2536.03}

rod_2024 = rod[rod["leto"] == 2024][["regija", "na1000"]].rename(columns={"na1000": "rodnost_2024"})

df_merge = (
    df[~df["regija"].isin(exclude)]
    .merge(rod_2024, on="regija")
    .sort_values("bruto_2023", ascending=True)
    .reset_index(drop=True)
)

V spodnji analizi raziskujemo povezanost med ekonomsko blaginjo, merjeno s povprečno bruto plačo (2023), in demografskimi trendi, natančneje rodnostjo (2024), po slovenskih statističnih regijah. Cilj je ugotoviti, ali višji prihodki v regiji delujejo kot spodbuden dejavnik za večje število rojstev na 1.000 prebivalcev.

In [2]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_merge["bruto_2023"], y=df_merge["regija"],
    orientation="h",
    marker_color=COLORS["sandy_brown"], marker_line_width=0,
    name="Bruto plača 2023 (€)",
    text=[f"{v:.0f} €" for v in df_merge["bruto_2023"]],
    textposition="outside", textfont=dict(size=10, color="#333333"),
    xaxis="x1",
    hovertemplate="<b>%{y}</b><br>Bruto plača: %{x:.0f} €<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=df_merge["rodnost_2024"], y=df_merge["regija"],
    mode="markers+text",
    marker=dict(size=14, color=COLORS["blushed_brick"], symbol="diamond",
                line=dict(color="white", width=1.5)),
    text=[f"{v:.1f}" for v in df_merge["rodnost_2024"]],
    textposition="top center", textfont=dict(size=9, color="#333333"),
    name="Rodnost 2024 (na 1.000 preb.)",
    xaxis="x2",
    hovertemplate="<b>%{y}</b><br>Rodnost: %{x} ‰<extra></extra>"
))

fig.add_vline(
    x=SLO_BRUTO[2023], line_dash="dash",
    line_color=COLORS["muted_olive"], line_width=2,
    annotation_text=f"SLO povprečje {SLO_BRUTO[2023]:.0f} €",
    annotation_position="top right",
    annotation_font=dict(color=COLORS["muted_olive"], size=11)
)

fig.update_layout(
    title="Rangiranje regij po bruto plači (2023) in rodnosti (2024)",
    height=580, plot_bgcolor="#FAFAF8", paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=12, color="#333333"),
    title_font=dict(size=15, color="#333333"),
    xaxis=dict(title="Bruto plača (EUR)", range=[1800, 3300],
               gridcolor="#E5E5E0", showline=True, linecolor="#CCCCCC",
               title_font=dict(size=12, color="#333333"), tickfont=dict(color="#333333")),
    xaxis2=dict(title="Novorojeni na 1.000 prebivalcev", overlaying="x",
                side="top", range=[5, 14], showgrid=False,
                tickcolor=COLORS["blushed_brick"],
                title_font=dict(color=COLORS["blushed_brick"], size=12),
                tickfont=dict(color=COLORS["blushed_brick"])),
    yaxis=dict(tickfont=dict(size=11, color="#333333")),
    legend=dict(x=0.6, y=0.05, bgcolor="rgba(255,255,255,0.9)",
                bordercolor="#DDDDDD", borderwidth=1, font=dict(color="#333333")),
    bargap=0.3, margin=dict(l=160, r=40, t=80, b=60),
)

fig.show()

Vizualno lahko opazimo, da regije z višjimi plačami (npr. Jugovzhodna Slovenija in Osrednjeslovenska) pogosto izkazujejo tudi višje stopnje rodnosti, medtem ko imajo regije na dnu plačilne lestvice (npr. Pomurska, Obalno-kraška) rodnost pod povprečjem.

Omejitev podatkov: Analiza se osredotoča na primerjavo med letoma 2023 in 2024, saj so bili podatki za bruto plače na voljo za obdobje 2023–2025, medtem ko so se uradni podatki o rodnosti v času analize zaključili z letom 2024. Zaradi tega smo korelativni vpliv lahko preverili le za omenjeni presek let, kjer so podatki za obe spremenljivki popolni.

Za objektivno oceno povezanosti smo izračunali Pearsonov in Spearmanov koeficient korelacije:

In [3]:
r, p = stats.pearsonr(df_merge["bruto_2023"], df_merge["rodnost_2024"])
r_sp, p_sp = stats.spearmanr(df_merge["bruto_2023"], df_merge["rodnost_2024"])

smer = "pozitivna" if r > 0 else "negativna"
moc = "šibka" if abs(r) < 0.3 else "zmerna" if abs(r) < 0.6 else "močna"
sig = "statistično značilna (p < 0,05)" if p < 0.05 else "ni statistično značilna (p ≥ 0,05)"

print(f"Pearsonov r  = {r:.3f}   (p = {p:.3f})")
print(f"Spearmanov ρ = {r_sp:.3f}   (p = {p_sp:.3f})")
print()
print(f"Med povprečno bruto plačo regij (2023) in rodnostjo (2024) obstaja")
print(f"{moc} {smer} korelacija (r = {r:.3f}), ki {sig}.")

Pearsonov r  = 0.507   (p = 0.092)
Spearmanov ρ = 0.575   (p = 0.050)

Med povprečno bruto plačo regij (2023) in rodnostjo (2024) obstaja
zmerna pozitivna korelacija (r = 0.507), ki ni statistično značilna (p ≥ 0,05).


**Smer in moč**: Med povprečno bruto plačo in rodnostjo obstaja zmerna pozitivna korelacija. To pomeni, da z naraščanjem plač v regiji načeloma narašča tudi rodnost.

**Statistična značilnost**: Čeprav je trend opazen, rezultat ni statistično značilen ($p > 0,05$). To je posledica majhnega vzorca (le 12 statističnih regij). Za potrditev trdne vzročne povezave bi potrebovali daljše časovno obdobje ali podrobnejše podatke na ravni občin.

**Izstopajoči primeri**: Jugovzhodna Slovenija močno izstopa z najvišjo rodnostjo ($9,5$), kar nakazuje, da na demografsko sliko poleg plač vplivajo še drugi specifični lokalni dejavniki.